In [1]:
# TODO check the intersection between croplands from SPAM and MODIS land classes.
# It's possible that there will be gaps to fill.

# Creating an inventory of feed (energy, protein, NDF)

- We have CSVs for each production system, each of which has 832,827 rows, and columns for point in time production values for each crop. - - These data points can be spatially mapped. 
- Our first task will be to turn these CSVs into raster files for each crop.
- We'll then use those raster values to get dry crop residue mass, energy, protein and NDF, all on a per-pixel basis

### Translate CSVs into raster files

In [5]:
import pandas as pd

IRRIGATED_PROD_CSV = "../../data/processed/production_pit_i.csv"
RAINFED_HIGH_INPUTS_PROD_CSV = "../../data/processed/production_pit_h.csv"
RAINFED_LOW_INPUTS_PROD_CSV = "../../data/processed/production_pit_l.csv"
SUBSISTENCE_PROD_CSV = "../../data/processed/production_pit_s.csv"
SPAM_CROP_LIST_CSV = "../../data/external/spam_crop_list.csv"

In [6]:
spam_croplist = pd.read_csv(SPAM_CROP_LIST_CSV)
spam_crop_mappings = list(zip(list(spam_croplist.spam_short), list(spam_croplist.spam_long)))

i_df = pd.read_csv(IRRIGATED_PROD_CSV)
h_df = pd.read_csv(RAINFED_HIGH_INPUTS_PROD_CSV)
l_df = pd.read_csv(RAINFED_LOW_INPUTS_PROD_CSV)
s_df = pd.read_csv(SUBSISTENCE_PROD_CSV)

In [169]:
# We should be able to take the x,y centroid values and convert them to an equivalent 
# raster indec on our (2160, 4320) raster. Using this method, we can "stamp" the CSV
# values onto an empty raster grid.

from math import floor
from tqdm import tqdm

# Function for converting from coordinates into raster indices

y_grid_cells_per_degree = 2160 / 180
x_grid_cells_per_degree = 4320 / 360

def coords_to_grid_idx(x, y):
    x = x + 180
    y = y + 90
    
    x_idx = 2160 - (y * y_grid_cells_per_degree)  # flipped due to array indexing convention
    y_idx = x * x_grid_cells_per_degree
    
    return (floor(x_idx), floor(y_idx))

# Initialise empty rasters for each crop
wheat_arr = np.zeros((2160, 4320))
rice_arr = np.zeros((2160, 4320))
maize_arr = np.zeros((2160, 4320))
barley_arr = np.zeros((2160, 4320))
pearlmill_arr = np.zeros((2160, 4320))
smallmill_arr = np.zeros((2160, 4320))
sorghum_arr = np.zeros((2160, 4320))
oth_cereal_arr = np.zeros((2160, 4320))
potato_arr = np.zeros((2160, 4320))
sweet_pot_arr = np.zeros((2160, 4320))
yams_arr = np.zeros((2160, 4320))
cassava_arr = np.zeros((2160, 4320))
oth_root_arr = np.zeros((2160, 4320))
bean_arr = np.zeros((2160, 4320))
chickpea_arr = np.zeros((2160, 4320))
cowpea_arr = np.zeros((2160, 4320))
pigeonpea_arr = np.zeros((2160, 4320))
lentil_arr = np.zeros((2160, 4320))
oth_pulse_arr = np.zeros((2160, 4320))
soybean_arr = np.zeros((2160, 4320))
groundnut_arr = np.zeros((2160, 4320))
coconut_arr = np.zeros((2160, 4320))
oilpalm_arr = np.zeros((2160, 4320))
sunflower_arr = np.zeros((2160, 4320))
rapeseed_arr = np.zeros((2160, 4320))
sesameseed_arr = np.zeros((2160, 4320))
oth_oil_arr = np.zeros((2160, 4320))
sugarcane_arr = np.zeros((2160, 4320))
sugarbeet_arr = np.zeros((2160, 4320))
cotton_arr = np.zeros((2160, 4320))
oth_fibre_arr = np.zeros((2160, 4320))
ara_coffee_arr = np.zeros((2160, 4320))
rob_coffee_arr = np.zeros((2160, 4320))
cocoa_arr = np.zeros((2160, 4320))
tea_arr = np.zeros((2160, 4320))
tobacco_arr = np.zeros((2160, 4320))
banana_arr = np.zeros((2160, 4320))
plantain_arr = np.zeros((2160, 4320))
trop_fruit_arr = np.zeros((2160, 4320))
temp_fruit_arr = np.zeros((2160, 4320))
vegetable_arr = np.zeros((2160, 4320))
rest_crop_arr = np.zeros((2160, 4320))

# Combine all our point in time production values for a given crop across
# the different systems (i, h, l, s) into individual rasters:
for idx in tqdm(range(len(i_df))):
    
    row_i = i_df.loc[idx]
    row_h = h_df.loc[idx]
    row_l = l_df.loc[idx]
    row_s = s_df.loc[idx]
    
    grid_idx = coords_to_grid_idx(row_i.x, row_i.y)
    
    for row in [row_i, row_h, row_l, row_s]:
        
        wheat_arr[grid_idx] += row.wheat_pit
        rice_arr[grid_idx] += row.rice_pit
        maize_arr[grid_idx] += row.maize_pit
        barley_arr[grid_idx] += row.barley_pit
        pearlmill_arr[grid_idx] += row.pearlmill_pit
        smallmill_arr[grid_idx] += row.smallmill_pit
        sorghum_arr[grid_idx] += row.sorghum_pit
        oth_cereal_arr[grid_idx] += row.oth_cereal_pit
        potato_arr[grid_idx] += row.potato_pit
        sweet_pot_arr[grid_idx] += row.sweet_pot_pit
        yams_arr[grid_idx] += row.yams_pit
        cassava_arr[grid_idx] += row.cassava_pit
        oth_root_arr[grid_idx] += row.oth_root_pit
        bean_arr[grid_idx] += row.bean_pit
        chickpea_arr[grid_idx] += row.chickpea_pit
        cowpea_arr[grid_idx] += row.cowpea_pit
        pigeonpea_arr[grid_idx] += row.pigeonpea_pit
        lentil_arr[grid_idx] += row.lentil_pit
        oth_pulse_arr[grid_idx] += row.oth_pulse_pit
        soybean_arr[grid_idx] += row.soybean_pit
        groundnut_arr[grid_idx] += row.groundnut_pit
        coconut_arr[grid_idx] += row.coconut_pit
        oilpalm_arr[grid_idx] += row.oilpalm_pit
        sunflower_arr[grid_idx] += row.sunflower_pit
        rapeseed_arr[grid_idx] += row.rapeseed_pit
        sesameseed_arr[grid_idx] += row.sesameseed_pit
        oth_oil_arr[grid_idx] += row.oth_oil_pit
        sugarcane_arr[grid_idx] += row.sugarcane_pit
        sugarbeet_arr[grid_idx] += row.sugarbeet_pit
        cotton_arr[grid_idx] += row.cotton_pit
        oth_fibre_arr[grid_idx] += row.oth_fibre_pit
        ara_coffee_arr[grid_idx] += row.ara_coffee_pit
        rob_coffee_arr[grid_idx] += row.rob_coffee_pit
        cocoa_arr[grid_idx] += row.cocoa_pit
        tea_arr[grid_idx] += row.tea_pit
        tobacco_arr[grid_idx] += row.tobacco_pit
        banana_arr[grid_idx] += row.banana_pit
        plantain_arr[grid_idx] += row.plantain_pit
        trop_fruit_arr[grid_idx] += row.trop_fruit_pit
        temp_fruit_arr[grid_idx] += row.temp_fruit_pit
        vegetable_arr[grid_idx] += row.vegetable_pit
        rest_crop_arr[grid_idx] += row.rest_crop_pit    

100%|██████████| 832827/832827 [16:27<00:00, 843.54it/s]


In [171]:
# Save the results

import rasterio

RASTER_FOR_TRANSFORM = '../../data/raw/SPAM_2010/spam2010v2r0_global_prod.geotiff/spam2010V2r0_global_P_ACOF_A.tif'
transform_dataset = rasterio.open(RASTER_FOR_TRANSFORM)

OUTPUT_FOLDER = "../../data/processed/production_pit"

datasets_config = [
    (wheat_arr, "wheat"),
    (rice_arr, "rice"),
    (maize_arr, "maize"),
    (barley_arr, "barley"),
    (pearlmill_arr, "pearlmill"),
    (smallmill_arr, "smallmill"),
    (sorghum_arr, "sorghum"),
    (oth_cereal_arr, "oth_cereal"),
    (potato_arr, "potato"),
    (sweet_pot_arr, "sweet_pot"),
    (yams_arr, "yams"),
    (cassava_arr, "cassava"),
    (oth_root_arr, "oth_root"),
    (bean_arr, "bean"),
    (chickpea_arr, "chickpea"),
    (cowpea_arr, "cowpea"),
    (pigeonpea_arr, "pigeonpea"),
    (lentil_arr, "lentil"),
    (oth_pulse_arr, "oth_pulse"),
    (soybean_arr, "soybean"),
    (groundnut_arr, "groundnut"),
    (coconut_arr, "coconut"),
    (oilpalm_arr, "oilpalm"),
    (sunflower_arr, "sunflower"),
    (rapeseed_arr, "rapeseed"),
    (sesameseed_arr, "sesameseed"),
    (oth_oil_arr, "oth_oil"),
    (sugarcane_arr, "sugarcane"),
    (sugarbeet_arr, "sugarbeet"),
    (cotton_arr, "cotton"),
    (oth_fibre_arr, "oth_fibre"),
    (ara_coffee_arr, "ara_coffee"),
    (rob_coffee_arr, "rob_coffee"),
    (cocoa_arr, "cocoa"),
    (tea_arr, "tea"),
    (tobacco_arr, "tobacco"),
    (banana_arr, "banana"),
    (plantain_arr, "plantain"),
    (trop_fruit_arr, "trop_fruit"),
    (temp_fruit_arr, "temp_fruit"),
    (vegetable_arr, "vegetable"),
    (rest_crop_arr, "rest_crop"),
]

for config in datasets_config:
    
    arr = config[0]
    crop_name = config[1]

    dataset = rasterio.open(
        f"{OUTPUT_FOLDER}/{crop_name}_prod_pit.tif",
        "w",
        driver="GTiff",
        height=arr.shape[0],
        width=arr.shape[1],
        count=1,
        dtype=arr.dtype,
        crs='+proj=latlong',
        transform=transform_dataset.transform
    )

    dataset.write(arr, 1)
    dataset.close()




### Resample grassland data

In [8]:
# Define a re-sampling method

import rasterio
from rasterio import warp
from rasterio.enums import Resampling
import numpy as np


def resample_raster_file(src_file, resampling_method = Resampling.sum, upscale_factor=0.1):

    with rasterio.open(src_file) as dataset:
        
        arr = dataset.read(1)
        
        data, transform = warp.reproject(
            arr,
            resampling=resampling_method,
            dst_resolution=(int(dataset.height * upscale_factor), int(dataset.width * upscale_factor))
        )
        
        # print("Reading and resampling...")
        # resample data to target shape
        # data = dataset.read(
        #     out_shape=(
        #         dataset.count,
        #         int(dataset.height * upscale_factor),
        #         int(dataset.width * upscale_factor)
        #     ),
        #     resampling=resampling_method
        # )
        
#         print("Scaling the transform...")
#         # scale image transform
#         transform = dataset.transform * dataset.transform.scale(
#             (dataset.width / data.shape[-1]),
#             (dataset.height / data.shape[-2])
#         )
        
#         print("Done")
        
        return data, transform


BIOMASS_FROM_GRASSLAND_RASTER_FILE = f"../../data/processed/production_pit/biomass_from_grassland_tonnes.tif"
biomass_from_grassland_dataset = rasterio.open(BIOMASS_FROM_GRASSLAND_RASTER_FILE)
biomass_from_grassland_arr = biomass_from_grassland_dataset.read(1)

RESAMPLED_RASTER = "../../data/processed/production_pit/biomass_from_grassland_tonnes_resampled.tif"
DOWNSCALE_FACTOR = 7200 / 4320

# Re-sample the biomass raster to crop raster (down-scale by a factor of 18)
resampled_biomass_arr, resampled_biomass_transform = resample_raster_file(
    BIOMASS_FROM_GRASSLAND_RASTER_FILE, 
    resampling_method=Resampling.sum, 
    upscale_factor=1/DOWNSCALE_FACTOR
)

# Save processed data
resampled_biomass_dataset = rasterio.open(
    RESAMPLED_RASTER,
    "w",
    driver="GTiff",
    height=biomass_from_grassland_arr.shape[0] / DOWNSCALE_FACTOR,
    width=biomass_from_grassland_arr.shape[1] / DOWNSCALE_FACTOR,
    count=1,
    dtype=resampled_biomass_arr.dtype,
    crs='+proj=latlong',
    transform=resampled_biomass_transform
)
resampled_biomass_arr = np.squeeze(resampled_biomass_arr)
resampled_biomass_dataset.write(resampled_biomass_arr, 1)
resampled_biomass_dataset.close()

AttributeError: 'NoneType' object has no attribute 'xoff'

In [4]:
biomass_from_grassland_arr.sum()

492280897474.3974

In [5]:
resampled_biomass_arr.sum()

177221123110.0083

### Convert to energy and protein

In [42]:
import rasterio
import pandas as pd
import numpy as np
from tqdm import tqdm
    
OUTPUT_FOLDER = "../../data/processed"
MACRO_TABLE_PATH = "../../data/processed/lookup_tables/spam_crop_macro_lookup.csv"
macro_table = pd.read_csv(MACRO_TABLE_PATH, index_col="spam_crop_long")

RASTER_FOR_TRANSFORM = '../../data/raw/SPAM_2010/spam2010v2r0_global_prod.geotiff/spam2010V2r0_global_P_ACOF_A.tif'
transform_dataset = rasterio.open(RASTER_FOR_TRANSFORM)

crops = ["wheat", "rice", "maize", "barley", "pearlmill", "smallmill", "sorghum", 
         "oth_cereal", "potato", "sweet_pot", "yams", "cassava", "oth_root", "bean", 
         "chickpea", "cowpea", "pigeonpea", "lentil", "oth_pulse", "soybean", "groundnut", 
         "coconut", "oilpalm", "sunflower", "rapeseed", "sesameseed", "oth_oil", 
         "sugarcane", "sugarbeet", "cotton", "oth_fibre",
         "banana", "plantain", "trop_fruit", "temp_fruit", 
         "vegetable", "rest_crop"] # exclude tobacco, tea, ara_coffee, rob_coffee, cocoa


# Let's save an array for protein and energy for each crop, and then a total protein and a
# total energy array
total_protein_arr = np.zeros((2160, 4320))
total_energy_arr = np.zeros((2160, 4320))

for crop_name in tqdm(crops):
    
    dataset = rasterio.open(f"{OUTPUT_FOLDER}/production_pit/{crop_name}_prod_pit.tif")
    arr = dataset.read(1)
    crop_data = macro_table.loc[crop_name]
    dry_res_arr = arr * crop_data.harvest_dm_fraction * crop_data.dry_rpr
    
    energy_arr = dry_res_arr * crop_data.ruminant_me * 1000 # 1000 is to convert MJ/kg to MJ/tonne
    total_energy_arr = total_energy_arr + energy_arr
    
    protein_arr = dry_res_arr * (crop_data.crude_protein / 100) # crude protein is % of mass, so divide by 100
    
    total_protein_arr = total_protein_arr + protein_arr
    
    protein_dataset = rasterio.open(
        f"{OUTPUT_FOLDER}/protein/{crop_name}_protein.tif",
        "w",
        driver="GTiff",
        height=protein_arr.shape[0],
        width=protein_arr.shape[1],
        count=1,
        dtype=protein_arr.dtype,
        crs='+proj=latlong',
        transform=transform_dataset.transform
    )

    protein_dataset.write(protein_arr, 1)
    protein_dataset.close()
    
    energy_dataset = rasterio.open(
        f"{OUTPUT_FOLDER}/energy/{crop_name}_energy.tif",
        "w",
        driver="GTiff",
        height=energy_arr.shape[0],
        width=energy_arr.shape[1],
        count=1,
        dtype=energy_arr.dtype,
        crs='+proj=latlong',
        transform=transform_dataset.transform
    )

    energy_dataset.write(energy_arr, 1)
    energy_dataset.close()

    
# Now do grasses
dataset = rasterio.open(f"../../data/processed/production_pit/biomass_from_grassland_tonnes_resampled.tif")
arr = dataset.read(1)
grass_data = macro_table.loc["Grassland"]

dry_res_arr = arr * grass_data.harvest_dm_fraction * grass_data.dry_rpr
energy_arr = dry_res_arr * grass_data.ruminant_me * 1000 # 1000 is to convert MJ/kg to MJ/tonne
total_energy_arr = total_energy_arr + energy_arr

protein_arr = dry_res_arr * (grass_data.crude_protein / 100) # crude protein is % of mass, so divide by 100
total_protein_arr = total_protein_arr + protein_arr

protein_dataset = rasterio.open(
    f"{OUTPUT_FOLDER}/protein/grassland_protein.tif",
    "w",
    driver="GTiff",
    height=protein_arr.shape[0],
    width=protein_arr.shape[1],
    count=1,
    dtype=protein_arr.dtype,
    crs='+proj=latlong',
    transform=transform_dataset.transform
)

protein_dataset.write(protein_arr, 1)
protein_dataset.close()

energy_dataset = rasterio.open(
    f"{OUTPUT_FOLDER}/energy/grassland_energy.tif",
    "w",
    driver="GTiff",
    height=energy_arr.shape[0],
    width=energy_arr.shape[1],
    count=1,
    dtype=energy_arr.dtype,
    crs='+proj=latlong',
    transform=transform_dataset.transform
)
energy_dataset.write(energy_arr, 1)
energy_dataset.close()


# Write totals
total_protein_dataset = rasterio.open(
    f"{OUTPUT_FOLDER}/protein/total_protein.tif",
    "w",
    driver="GTiff",
    height=total_protein_arr.shape[0],
    width=total_protein_arr.shape[1],
    count=1,
    dtype=total_protein_arr.dtype,
    crs='+proj=latlong',
    transform=transform_dataset.transform
)
total_protein_dataset.write(total_protein_arr, 1)
total_protein_dataset.close()

total_energy_dataset = rasterio.open(
    f"{OUTPUT_FOLDER}/energy/total_energy.tif",
    "w",
    driver="GTiff",
    height=total_energy_arr.shape[0],
    width=total_energy_arr.shape[1],
    count=1,
    dtype=total_energy_arr.dtype,
    crs='+proj=latlong',
    transform=transform_dataset.transform
)
total_energy_dataset.write(total_energy_arr, 1)
total_energy_dataset.close()
    

100%|██████████| 37/37 [00:08<00:00,  4.51it/s]


In [43]:
np.nansum(dry_res_arr)

177221123110.00098

In [10]:
macro_table.loc["wheat"]

full_name                                        Wheat
group                                          cereals
2010_pit_prod                              592510726.8
harvest_dm_fraction                               0.88
2010_pit_prod_dry                          521409439.5
dry_rpr                                            1.0
crude_protein                                      4.2
ruminant_me                                        6.8
ndf                                               77.5
2010_pit_residues_dry (tonnes)             521409439.5
2010_pit_ruminant_me_dry (MJ)          3545584188896.0
2010_pit_crude_protein_dry (tonnes)        21899196.46
2010_pit_ruminant_ndf_dry (tonnes)         404092315.6
Name: wheat, dtype: object

In [30]:
np.nansum(total_energy_arr)

1538960901207192.8

In [31]:
np.nansum(total_protein_arr)

16559227466.768593